In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import zipfile

In [ ]:
!curl -L -o /content/archive.zip https://www.kaggle.com/api/v1/datasets/download/gpiosenka/sports-classification

In [ ]:
with zipfile.ZipFile("/content/archive.zip", 'r') as sports_zip:
  sports_zip.extractall("/content/SportsDataset")

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomResizedCrop((128, 128), scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

transform_data = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [ ]:
train_data = datasets.ImageFolder(root='/content/SportsDataset/train', transform=transform_train)
test_data = datasets.ImageFolder(root='/content/SportsDataset/test', transform=transform_data)

In [ ]:
sports = train_data.classes
sports

In [ ]:
len(sports)

In [ ]:
train = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
test = DataLoader(test_data, batch_size=64, num_workers=2)

In [ ]:
image, label = next(iter(train))
plt.figure(figsize=(8, 10))
for i in range(20):
  plt.subplot(4, 5, i+1)
  plt.imshow(image[i].permute(1, 2, 0))
  plt.title(f'{sports[label[i]]}', fontsize=8)
  plt.axis('off')
  plt.tight_layout()
plt.show()

In [ ]:
class CheckSportsModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.first = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(128, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(256, 512, kernel_size=3, padding=1),
        nn.BatchNorm2d(512),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d((1, 1))
    )
    self.second = nn.Sequential(
        nn.Flatten(),
        nn.Dropout(0.3),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100)
    )
  def forward(self, x):
    x = self.first(x)
    x = self.second(x)
    return x

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
sports_model = CheckSportsModel().to(device)

In [ ]:
loss_fn_sports = nn.CrossEntropyLoss(label_smoothing=0.1)
sports_optimizer = optim.Adam(sports_model.parameters(), lr=0.001)
sports_scheduler = optim.lr_scheduler.CosineAnnealingLR(sports_optimizer, T_max=40)

In [ ]:
for epoch in range(40):
  sports_model.train()
  total_loss = 0

  for x_batch, y_batch in train:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)

    y_pred = sports_model(x_batch)
    loss = loss_fn_sports(y_pred, y_batch)

    sports_optimizer.zero_grad()
    loss.backward()
    sports_optimizer.step()

    total_loss += loss.item()
  sports_scheduler.step()
  print(f'Эпоха: {epoch+1}, Потери: {round(total_loss, 2)}')

In [ ]:
sports_model.eval()
correct, total = 0, 0

with torch.no_grad():
  for x_batch, y_batch in test:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)

    y_pred = sports_model(x_batch)
    predicted = torch.argmax(y_pred, dim=1)

    total += y_batch.size(0)
    correct += (predicted == y_batch).sum().item()

accuracy = 100 * correct / total
print(f'Точность модели на тестовых данных: {round(accuracy, 2)}%')

In [ ]:
torch.save(sports_model.state_dict(), 'sports_model.pth')
torch.save(sports, 'sports_classes.pth')